# 02 Baseline Financial Segment Model

本 notebook 读取第一个 notebook 生成的 company-level latest T1 训练集，建立可解释的 financial segment baseline。

优先任务：

```text
Task A: BB vs non_BB
Task B: BB / SME / MidCorporate multiclass sanity check
```

建模原则：

- 只使用 financial proxy、field count、evidence tier、account category、sector 等特征；
- 不使用 `turnover`、`latest_observed_turnover` 或任何 turnover-derived numeric 字段作为特征；
- label 仅来自 latest filing 的 observed turnover；
- 输出模型评估、预测结果、feature importance / coefficient，便于报告解释。


输出说明：

```text
Predicted segment = argmax(P(BB), P(SME), P(MidCorporate))
Predicted BB label = argmax(P(BB), P(non_BB))
```

因此输出中同时保留最终预测类别和各类别概率。


In [ ]:
from pathlib import Path
import json
import os
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text

warnings.filterwarnings("ignore", category=UserWarning)


def find_repo_root():
    env_root = os.environ.get("PROJECT_ROOT")
    starts = []
    if env_root:
        starts.append(Path(env_root))
    starts.append(Path.cwd())

    for start in starts:
        start = start.resolve()
        for p in [start, *start.parents]:
            if (p / ".git").exists() and (p / "00 Data Preparation + EDA").exists():
                return p
    raise FileNotFoundError(
        "Cannot find repository root. Start Jupyter from inside the cloned repo, "
        "or set PROJECT_ROOT to the repository root."
    )


def require_exists(path, label="path"):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing {label}: {path}")
    return path


def first_existing(paths, label):
    paths = [Path(path) for path in paths]
    for path in paths:
        if path.exists():
            return path
    tried = "\n".join(f"- {path}" for path in paths)
    raise FileNotFoundError(f"Cannot find {label}. Tried:\n{tried}")


PROJECT_ROOT = find_repo_root()
FINANCIAL_DIR = PROJECT_ROOT / "00 Data Preparation + EDA" / "Financial Data"
SMALL_TEST_DIR = FINANCIAL_DIR / "Small Scale Data Test"
SAMPLE_COMPANIES_DIR = SMALL_TEST_DIR / "Sample Companies"
LOCAL_DATA_ROOT = Path(os.environ.get(
    "LOCAL_DATA_ROOT",
    r"E:\000硕士毕设\财务数据\Local Large Data",
))
ACCOUNTS_ZIP_DIR = first_existing(
    [
        LOCAL_DATA_ROOT / "Accounts Data_2025.1_2026.5",
        FINANCIAL_DIR / "Accounts Data_2025.1_2026.5",
    ],
    "accounts ZIP directory",
)
BULK_MATCH_DIR = SMALL_TEST_DIR / "2025全年bulk匹配"
MODEL_PREP_DIR = BULK_MATCH_DIR / "模型训练与数据处理"
MODEL_TRAIN_DIR = MODEL_PREP_DIR / "模型训练"

require_exists(FINANCIAL_DIR, "financial data directory")
require_exists(SMALL_TEST_DIR, "small scale test directory")
require_exists(ACCOUNTS_ZIP_DIR, "accounts ZIP directory")


BASE_DIR = MODEL_PREP_DIR
MODEL_DIR = MODEL_TRAIN_DIR
MODEL_DIR.mkdir(parents=True, exist_ok=True)

INPUT_T1 = BASE_DIR / "financial_features_candidate_latest_model_ready_t1.csv"
INPUT_LATEST = BASE_DIR / "financial_features_candidate_latest_company.csv"
FEATURE_CONFIG = BASE_DIR / "model_feature_columns.json"

OUTPUT_BB_METRICS = MODEL_DIR / "baseline_bb_vs_non_bb_metrics.csv"
OUTPUT_BB_CV = MODEL_DIR / "baseline_bb_vs_non_bb_cross_validation.csv"
OUTPUT_BB_PREDICTIONS = MODEL_DIR / "baseline_bb_vs_non_bb_predictions.csv"
OUTPUT_BB_FEATURES = MODEL_DIR / "baseline_bb_vs_non_bb_feature_importance.csv"
OUTPUT_BB_TREE_RULES = MODEL_DIR / "baseline_bb_vs_non_bb_decision_tree_rules.txt"

OUTPUT_MULTI_METRICS = MODEL_DIR / "baseline_multiclass_metrics.csv"
OUTPUT_MULTI_CV = MODEL_DIR / "baseline_multiclass_cross_validation.csv"
OUTPUT_MULTI_PREDICTIONS = MODEL_DIR / "baseline_multiclass_predictions.csv"
OUTPUT_MULTI_FEATURES = MODEL_DIR / "baseline_multiclass_feature_importance.csv"
OUTPUT_MULTI_TREE_RULES = MODEL_DIR / "baseline_multiclass_decision_tree_rules.txt"

RANDOM_STATE = 42
TEST_SIZE = 0.25

require_exists(INPUT_T1, "T1 model-ready file")
require_exists(FEATURE_CONFIG, "feature config")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("LOCAL_DATA_ROOT:", LOCAL_DATA_ROOT)
print("Input T1:", INPUT_T1)
print("Feature config:", FEATURE_CONFIG)
print("Output dir:", MODEL_DIR)


## 1. 读取 company-level T1 训练集


In [ ]:
if not INPUT_T1.exists():
    raise FileNotFoundError(
        f"{INPUT_T1} not found. Please run 01_build_company_level_latest_features.ipynb first."
    )

df = pd.read_csv(INPUT_T1, dtype={"CompanyNumber": str})
df["CompanyNumber"] = df["CompanyNumber"].astype(str).str.upper().str.zfill(8)

with FEATURE_CONFIG.open("r", encoding="utf-8") as f:
    feature_config = json.load(f)

print("Rows:", len(df))
print("Unique companies:", df["CompanyNumber"].nunique())
display(df[[
    "CompanyNumber",
    "CompanyName",
    "latest_observed_turnover",
    "latest_observed_segment_from_turnover",
    "model_label_bb_vs_non_bb",
    "financial_core_score",
    "financial_conservative_score",
    "Accounts_AccountCategory",
    "primary_sector",
]].head())

display(df["latest_observed_segment_from_turnover"].value_counts(dropna=False).to_frame("companies"))
display(df["model_label_bb_vs_non_bb"].value_counts(dropna=False).to_frame("companies"))


## 2. 特征选择与 leakage check


In [ ]:
numeric_features = [c for c in feature_config["numeric_feature_columns"] if c in df.columns]
binary_features = [c for c in feature_config["binary_feature_columns"] if c in df.columns]
categorical_features = [c for c in feature_config["categorical_feature_columns"] if c in df.columns]

LEAKAGE_PATTERNS = [
    "turnover",
    "observed_segment",
    "model_label",
    "label_available",
]

def has_leakage_name(col):
    lower = col.lower()
    return any(pattern in lower for pattern in LEAKAGE_PATTERNS)

selected_numeric = [c for c in numeric_features if not has_leakage_name(c)]
selected_binary = [c for c in binary_features if not has_leakage_name(c)]
selected_categorical = [c for c in categorical_features if not has_leakage_name(c)]

feature_columns = selected_numeric + selected_binary + selected_categorical
leakage_removed = [
    c for c in numeric_features + binary_features + categorical_features
    if c not in feature_columns
]

print("Numeric features:", selected_numeric)
print("Binary features:", selected_binary)
print("Categorical features:", selected_categorical)
print("Removed by leakage check:", leakage_removed)
print("Total features before one-hot:", len(feature_columns))

X = df[feature_columns].copy()
for col in selected_binary:
    X[col] = X[col].map({True: 1, False: 0, "True": 1, "False": 0, "true": 1, "false": 0}).fillna(X[col]).astype(float)

y_bb = df["model_label_bb_vs_non_bb"].copy()
y_multi = df["latest_observed_segment_from_turnover"].copy()

display(X.head())


## 3. Preprocessing 与模型定义


In [ ]:
def make_onehot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

binary_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", make_onehot_encoder()),
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, selected_numeric),
        ("bin", binary_transformer, selected_binary),
        ("cat", categorical_transformer, selected_categorical),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

models = {
    "logistic_regression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        solver="lbfgs",
    ),
    "decision_tree_depth4": DecisionTreeClassifier(
        max_depth=4,
        min_samples_leaf=40,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
}

pipelines = {
    name: Pipeline(steps=[("preprocess", preprocess), ("model", model)])
    for name, model in models.items()
}

print(pipelines)


## 4. Helper functions


In [ ]:
def get_feature_names(fitted_pipeline):
    pre = fitted_pipeline.named_steps["preprocess"]
    try:
        return list(pre.get_feature_names_out())
    except Exception:
        names = []
        names.extend(selected_numeric)
        names.extend(selected_binary)
        cat_step = pre.named_transformers_["cat"].named_steps["onehot"]
        try:
            names.extend(list(cat_step.get_feature_names_out(selected_categorical)))
        except Exception:
            names.extend(selected_categorical)
        return names


def binary_metrics(y_true, pred, prob):
    positive = "BB"
    y_true_binary = (pd.Series(y_true).values == positive).astype(int)
    pred_binary = (pd.Series(pred).values == positive).astype(int)
    return {
        "accuracy": accuracy_score(y_true, pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "precision_BB": precision_score(y_true, pred, pos_label=positive, zero_division=0),
        "recall_BB": recall_score(y_true, pred, pos_label=positive, zero_division=0),
        "f1_BB": f1_score(y_true, pred, pos_label=positive, zero_division=0),
        "roc_auc_BB": roc_auc_score(y_true_binary, prob) if len(np.unique(y_true_binary)) == 2 else np.nan,
        "pr_auc_BB": average_precision_score(y_true_binary, prob) if len(np.unique(y_true_binary)) == 2 else np.nan,
    }


def multiclass_metrics(y_true, pred, prob, classes):
    out = {
        "accuracy": accuracy_score(y_true, pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "macro_f1": f1_score(y_true, pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, pred, average="weighted", zero_division=0),
    }
    try:
        out["roc_auc_ovr_weighted"] = roc_auc_score(y_true, prob, labels=classes, multi_class="ovr", average="weighted")
    except Exception:
        out["roc_auc_ovr_weighted"] = np.nan
    return out


def coefficient_or_importance_table(fitted_pipeline, model_name, task_name):
    names = get_feature_names(fitted_pipeline)
    model = fitted_pipeline.named_steps["model"]
    rows = []
    if hasattr(model, "coef_"):
        coef = model.coef_
        classes = list(getattr(model, "classes_", []))
        if coef.ndim == 1 or coef.shape[0] == 1:
            values = coef.ravel()
            class_name = classes[-1] if classes else ""
            rows = [
                {"task": task_name, "model": model_name, "class": class_name, "feature": n, "importance": v, "abs_importance": abs(v)}
                for n, v in zip(names, values)
            ]
        else:
            for class_name, class_coef in zip(classes, coef):
                rows.extend([
                    {"task": task_name, "model": model_name, "class": class_name, "feature": n, "importance": v, "abs_importance": abs(v)}
                    for n, v in zip(names, class_coef)
                ])
    elif hasattr(model, "feature_importances_"):
        rows = [
            {"task": task_name, "model": model_name, "class": "", "feature": n, "importance": v, "abs_importance": abs(v)}
            for n, v in zip(names, model.feature_importances_)
        ]
    return pd.DataFrame(rows).sort_values("abs_importance", ascending=False)


def add_prediction_id_columns(base_df, pred, prob_df):
    """Attach human-readable prediction columns while keeping raw probabilities.

    For binary BB screening:
      - predicted_bb_label is argmax(P(BB), P(non_BB))
      - predicted_bb_probability is P(BB), useful for threshold tuning

    For multiclass segment prediction:
      - predicted_segment is argmax(P(BB), P(SME), P(MidCorporate))
      - predicted_segment_probability is max segment probability
    """
    out = base_df[[
        "CompanyNumber",
        "CompanyName",
        "primary_sector",
        "Accounts_AccountCategory",
        "latest_observed_turnover",
        "latest_observed_segment_from_turnover",
        "model_label_bb_vs_non_bb",
        "financial_core_score",
        "financial_conservative_score",
        "available_core_balance_field_count",
        "financial_evidence_tier",
    ]].copy().reset_index(drop=True)

    prob_df = prob_df.reset_index(drop=True).copy()
    prob_cols = [c for c in prob_df.columns if c.startswith("prob_")]
    pred_series = pd.Series(pred, name="prediction").reset_index(drop=True)

    # Keep backward-compatible generic columns.
    out["prediction"] = pred_series
    out["predicted_label"] = pred_series
    out["predicted_label_probability"] = prob_df[prob_cols].max(axis=1)

    # Confidence gap between the largest and second largest class probability.
    sorted_probs = np.sort(prob_df[prob_cols].to_numpy(), axis=1)
    if sorted_probs.shape[1] >= 2:
        out["prediction_probability_margin"] = sorted_probs[:, -1] - sorted_probs[:, -2]
    else:
        out["prediction_probability_margin"] = np.nan

    if {"prob_BB", "prob_non_BB"}.issubset(prob_cols):
        out["predicted_bb_label"] = pred_series
        out["predicted_bb_probability"] = prob_df["prob_BB"]
        out["predicted_non_bb_probability"] = prob_df["prob_non_BB"]

    segment_prob_cols = [c for c in ["prob_BB", "prob_SME", "prob_MidCorporate"] if c in prob_cols]
    if len(segment_prob_cols) >= 2 and "prob_non_BB" not in prob_cols:
        out["predicted_segment"] = pred_series
        out["predicted_segment_probability"] = prob_df[segment_prob_cols].max(axis=1)

    return pd.concat([out, prob_df], axis=1)



## 5. Task A: BB vs non_BB baseline


In [ ]:
bb_mask = y_bb.isin(["BB", "non_BB"])
X_bb = X.loc[bb_mask].copy()
y_bb_task = y_bb.loc[bb_mask].copy()
df_bb = df.loc[bb_mask].copy()

X_train, X_test, y_train, y_test, df_train, df_test = train_test_split(
    X_bb,
    y_bb_task,
    df_bb,
    test_size=TEST_SIZE,
    stratify=y_bb_task,
    random_state=RANDOM_STATE,
)

bb_metric_rows = []
bb_feature_tables = []
bb_prediction_tables = []

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
bb_cv_rows = []

for model_name, pipe in pipelines.items():
    fold_metrics = []
    for fold, (train_idx, test_idx) in enumerate(cv.split(X_bb, y_bb_task), start=1):
        fold_model = pipe.fit(X_bb.iloc[train_idx], y_bb_task.iloc[train_idx])
        fold_pred = fold_model.predict(X_bb.iloc[test_idx])
        fold_classes = list(fold_model.named_steps["model"].classes_)
        fold_prob = fold_model.predict_proba(X_bb.iloc[test_idx])
        fold_prob_bb = fold_prob[:, fold_classes.index("BB")]
        metric = binary_metrics(y_bb_task.iloc[test_idx], fold_pred, fold_prob_bb)
        metric["fold"] = fold
        fold_metrics.append(metric)

    fold_df = pd.DataFrame(fold_metrics)
    for metric_name in ["balanced_accuracy", "f1_BB", "roc_auc_BB", "pr_auc_BB", "precision_BB", "recall_BB"]:
        bb_cv_rows.append({
            "task": "BB_vs_non_BB",
            "model": model_name,
            "metric": metric_name,
            "mean": float(fold_df[metric_name].mean()),
            "std": float(fold_df[metric_name].std()),
        })

    fitted = pipe.fit(X_train, y_train)
    pred = fitted.predict(X_test)
    classes = list(fitted.named_steps["model"].classes_)
    prob = fitted.predict_proba(X_test)
    bb_index = classes.index("BB")
    prob_bb = prob[:, bb_index]

    metrics = binary_metrics(y_test, pred, prob_bb)
    metrics.update({
        "task": "BB_vs_non_BB",
        "model": model_name,
        "train_rows": len(X_train),
        "test_rows": len(X_test),
        "test_BB_count": int((y_test == "BB").sum()),
        "test_non_BB_count": int((y_test == "non_BB").sum()),
    })
    bb_metric_rows.append(metrics)

    prob_df = pd.DataFrame({f"prob_{cls}": prob[:, i] for i, cls in enumerate(classes)})
    pred_table = add_prediction_id_columns(df_test, pred, prob_df)
    pred_table["task"] = "BB_vs_non_BB"
    pred_table["model"] = model_name
    bb_prediction_tables.append(pred_table)

    fi = coefficient_or_importance_table(fitted, model_name, "BB_vs_non_BB")
    bb_feature_tables.append(fi)

    if model_name.startswith("decision_tree"):
        tree_rules = export_text(
            fitted.named_steps["model"],
            feature_names=get_feature_names(fitted),
            max_depth=4,
        )
        OUTPUT_BB_TREE_RULES.write_text(tree_rules, encoding="utf-8")

bb_metrics = pd.DataFrame(bb_metric_rows)
bb_cv = pd.DataFrame(bb_cv_rows)
bb_predictions = pd.concat(bb_prediction_tables, ignore_index=True)
bb_features = pd.concat(bb_feature_tables, ignore_index=True)

display(bb_metrics)
display(bb_cv)

bb_display_cols = [
    "model",
    "CompanyNumber",
    "CompanyName",
    "model_label_bb_vs_non_bb",
    "predicted_bb_label",
    "predicted_bb_probability",
    "predicted_non_bb_probability",
    "predicted_label_probability",
    "prediction_probability_margin",
    "latest_observed_turnover",
    "latest_observed_segment_from_turnover",
    "financial_core_score",
    "financial_conservative_score",
]
display(bb_predictions[bb_display_cols].head(20))

bb_prediction_summary = (
    bb_predictions
    .groupby(["model", "model_label_bb_vs_non_bb", "predicted_bb_label"], dropna=False)
    .agg(
        rows=("CompanyNumber", "count"),
        median_prob_BB=("predicted_bb_probability", "median"),
        median_probability_margin=("prediction_probability_margin", "median"),
    )
    .reset_index()
)
display(bb_prediction_summary)
display(bb_features.head(30))
display(pd.DataFrame(confusion_matrix(y_test, pred, labels=["BB", "non_BB"]), index=["true_BB", "true_non_BB"], columns=["pred_BB", "pred_non_BB"]))


## 6. Task B: BB / SME / MidCorporate multiclass sanity check


In [ ]:
multi_classes = ["BB", "SME", "MidCorporate"]
multi_mask = y_multi.isin(multi_classes)
X_multi = X.loc[multi_mask].copy()
y_multi_task = y_multi.loc[multi_mask].copy()
df_multi = df.loc[multi_mask].copy()

X_train_m, X_test_m, y_train_m, y_test_m, df_train_m, df_test_m = train_test_split(
    X_multi,
    y_multi_task,
    df_multi,
    test_size=TEST_SIZE,
    stratify=y_multi_task,
    random_state=RANDOM_STATE,
)

multi_metric_rows = []
multi_feature_tables = []
multi_prediction_tables = []
multi_cv_rows = []

for model_name, pipe in pipelines.items():
    cv_scores = cross_validate(
        pipe,
        X_multi,
        y_multi_task,
        cv=cv,
        scoring={
            "balanced_accuracy": "balanced_accuracy",
            "f1_macro": "f1_macro",
            "f1_weighted": "f1_weighted",
        },
        n_jobs=None,
        error_score=np.nan,
    )
    for metric_name, values in cv_scores.items():
        if not metric_name.startswith("test_"):
            continue
        multi_cv_rows.append({
            "task": "multiclass_segment",
            "model": model_name,
            "metric": metric_name.replace("test_", ""),
            "mean": float(np.nanmean(values)),
            "std": float(np.nanstd(values)),
        })

    fitted = pipe.fit(X_train_m, y_train_m)
    pred = fitted.predict(X_test_m)
    classes = list(fitted.named_steps["model"].classes_)
    prob = fitted.predict_proba(X_test_m)

    metrics = multiclass_metrics(y_test_m, pred, prob, classes)
    metrics.update({
        "task": "multiclass_segment",
        "model": model_name,
        "train_rows": len(X_train_m),
        "test_rows": len(X_test_m),
    })
    multi_metric_rows.append(metrics)

    prob_df = pd.DataFrame({f"prob_{cls}": prob[:, i] for i, cls in enumerate(classes)})
    pred_table = add_prediction_id_columns(df_test_m, pred, prob_df)
    pred_table["task"] = "multiclass_segment"
    pred_table["model"] = model_name
    multi_prediction_tables.append(pred_table)

    fi = coefficient_or_importance_table(fitted, model_name, "multiclass_segment")
    multi_feature_tables.append(fi)

    if model_name.startswith("decision_tree"):
        tree_rules = export_text(
            fitted.named_steps["model"],
            feature_names=get_feature_names(fitted),
            max_depth=4,
        )
        OUTPUT_MULTI_TREE_RULES.write_text(tree_rules, encoding="utf-8")

multi_metrics = pd.DataFrame(multi_metric_rows)
multi_cv = pd.DataFrame(multi_cv_rows)
multi_predictions = pd.concat(multi_prediction_tables, ignore_index=True)
multi_features = pd.concat(multi_feature_tables, ignore_index=True)

display(multi_metrics)
display(multi_cv)

multi_display_cols = [
    "model",
    "CompanyNumber",
    "CompanyName",
    "latest_observed_segment_from_turnover",
    "predicted_segment",
    "predicted_segment_probability",
    "prediction_probability_margin",
    "prob_BB",
    "prob_SME",
    "prob_MidCorporate",
    "latest_observed_turnover",
    "financial_core_score",
    "financial_conservative_score",
]
display(multi_predictions[multi_display_cols].head(20))

multi_prediction_summary = (
    multi_predictions
    .groupby(["model", "latest_observed_segment_from_turnover", "predicted_segment"], dropna=False)
    .agg(
        rows=("CompanyNumber", "count"),
        median_predicted_segment_probability=("predicted_segment_probability", "median"),
        median_probability_margin=("prediction_probability_margin", "median"),
    )
    .reset_index()
)
display(multi_prediction_summary)
display(multi_features.head(40))
display(pd.DataFrame(confusion_matrix(y_test_m, pred, labels=multi_classes), index=[f"true_{c}" for c in multi_classes], columns=[f"pred_{c}" for c in multi_classes]))
print(classification_report(y_test_m, pred, labels=multi_classes, zero_division=0))


## 7. 写出模型结果


In [ ]:
bb_metrics.to_csv(OUTPUT_BB_METRICS, index=False, encoding="utf-8-sig")
bb_cv.to_csv(OUTPUT_BB_CV, index=False, encoding="utf-8-sig")
bb_predictions.to_csv(OUTPUT_BB_PREDICTIONS, index=False, encoding="utf-8-sig")
bb_features.to_csv(OUTPUT_BB_FEATURES, index=False, encoding="utf-8-sig")

multi_metrics.to_csv(OUTPUT_MULTI_METRICS, index=False, encoding="utf-8-sig")
multi_cv.to_csv(OUTPUT_MULTI_CV, index=False, encoding="utf-8-sig")
multi_predictions.to_csv(OUTPUT_MULTI_PREDICTIONS, index=False, encoding="utf-8-sig")
multi_features.to_csv(OUTPUT_MULTI_FEATURES, index=False, encoding="utf-8-sig")

run_summary = {
    "input_t1": str(INPUT_T1),
    "feature_config": str(FEATURE_CONFIG),
    "feature_columns": feature_columns,
    "numeric_features": selected_numeric,
    "binary_features": selected_binary,
    "categorical_features": selected_categorical,
    "rows": int(len(df)),
    "bb_vs_non_bb_counts": y_bb.value_counts(dropna=False).to_dict(),
    "multiclass_counts": y_multi.value_counts(dropna=False).to_dict(),
    "outputs": {
        "bb_metrics": str(OUTPUT_BB_METRICS),
        "bb_cv": str(OUTPUT_BB_CV),
        "bb_predictions": str(OUTPUT_BB_PREDICTIONS),
        "bb_features": str(OUTPUT_BB_FEATURES),
        "bb_tree_rules": str(OUTPUT_BB_TREE_RULES),
        "multi_metrics": str(OUTPUT_MULTI_METRICS),
        "multi_cv": str(OUTPUT_MULTI_CV),
        "multi_predictions": str(OUTPUT_MULTI_PREDICTIONS),
        "multi_features": str(OUTPUT_MULTI_FEATURES),
        "multi_tree_rules": str(OUTPUT_MULTI_TREE_RULES),
    },
}
(MODEL_DIR / "baseline_model_run_summary.json").write_text(
    json.dumps(run_summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Wrote:")
for path in [
    OUTPUT_BB_METRICS,
    OUTPUT_BB_CV,
    OUTPUT_BB_PREDICTIONS,
    OUTPUT_BB_FEATURES,
    OUTPUT_BB_TREE_RULES,
    OUTPUT_MULTI_METRICS,
    OUTPUT_MULTI_CV,
    OUTPUT_MULTI_PREDICTIONS,
    OUTPUT_MULTI_FEATURES,
    OUTPUT_MULTI_TREE_RULES,
    MODEL_DIR / "baseline_model_run_summary.json",
]:
    print(path)


## 8. 结果解释建议

报告中优先解释：

- `BB vs non_BB` 的 recall / precision / PR-AUC；
- `BB` 是否被系统性误判为 `SME`；
- `financial_core_score`、`financial_conservative_score`、field count、account category、sector 的重要性；
- `DecisionTree` 规则是否符合业务直觉；
- 模型是否只是学习了 disclosure pattern，而不是公司规模。

如果 `BB` recall 很低，下一步可以调整：

```text
class_weight
decision threshold
probability calibration
sector-specific baseline
account-category-specific baseline
```
